# LM Studio Llama 3.3 70B — `uni_subject` Annotation

**Model:** Llama 3.3 70B Instruct (Q4) via LM Studio

## 1. Imports & config

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os, time, re
from tqdm.auto import tqdm
from openai import OpenAI

LMSTUDIO_URL = "http://localhost:1234/v1"
MODEL_ID     = "meta/llama-3.3-70b"
TEMPERATURE  = 0
MAX_TOKENS   = 64
SEED         = 42

## 2. Set up client

In [2]:
client = OpenAI(base_url=LMSTUDIO_URL, api_key="lm-studio")
print("Models available in LM Studio:")
try:
    for m in client.models.list():
        print(f"  {m.id}")
except Exception as e:
    print(f"  Could not connect: {e}")

Models available in LM Studio:
  meta/llama-3.3-70b
  text-embedding-bge-m3
  qwen/qwen3-next-80b
  text-embedding-nomic-embed-text-v1.5
  deepseek-r1-distill-qwen-7b
  devstral-small-2505
  gemma-3-27b-it-qat
  openai/gpt-oss-20b


## 3. Load valid codes

In [3]:
from corex_eval import load_training_data

train_df = load_training_data(
    task="annotation",
    variable="uni_subject",
    features=["subject_label", "uni_subject"],
)
train_df = train_df.dropna(subset=["subject_label", "uni_subject"]).reset_index(drop=True)

valid_codes = sorted(train_df["uni_subject"].unique().tolist())
codes_str   = "\n".join(f"  {c}" for c in valid_codes)
print(f"{len(valid_codes)} valid subject codes")

/tmp/pycharm_project_967/corex_eval/silver.py:146: UserWarning: [silver] Dropped 2 row(s) for variable 'uni_subject' whose case_ids are not present in the gold standard.
  warnings.warn(


## 4. Build prompt & parser

In [4]:
SYSTEM_PROMPT = (
    "You are an expert political scientist coding university education data "
    "using the CoREx codebook.\n\n"
    "Task: given a subject description (combining degree type and field of study), "
    "assign the single most appropriate subject code.\n"
    "Respond with ONLY the exact code label as listed below — nothing else.\n\n"
    f"Valid subject codes:\n{codes_str}"
)

def parse_prediction(text, valid_codes):
    import re
    text = text.strip().strip("\"'")
    if text in valid_codes:
        return text
    code_map = {c.split(" = ", 1)[0].strip(): c for c in valid_codes if " = " in c}
    for code_num, full_code in code_map.items():
        if re.search(r"\\b" + re.escape(code_num) + r"\\b", text):
            return full_code
    text_lower = text.lower()
    for code in valid_codes:
        desc = code.split(" = ", 1)[1].lower() if " = " in code else code.lower()
        if desc in text_lower or text_lower in desc:
            return code
    return text

print(f"System prompt: {len(SYSTEM_PROMPT)} chars")

System prompt: 2504 chars


## 5. Load test inputs

In [5]:
from corex_eval import load_inputs

test_df = load_inputs(task="annotation", variable="uni_subject")
print(f"{len(test_df)} test rows")
test_df.head()

,case_id,spell_index,subject_label,uni_subject,workplace_label
0,002002,1,General Diploma of Management,"10 = Business, administration: Management, HRM",
1,002015,1,Doctorate Degree in Political Sciences,"04 = Social Sciences: Political sciences, inte...",
2,002021,1,Master of Science degrees at the University of...,11 = Law,
3,002022,1,graduated in law,11 = Law,
4,002030,1,Master of Science in Physics,"12 = Natural sciences: Biology, Biochemistry, ...",


predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
    subject = str(row["subject_label"] or "")
    user_msg = f"Subject description: {subject}"
    try:
        resp = client.chat.completions.create(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_msg},
            ],
            temperature=TEMPERATURE, max_tokens=MAX_TOKENS, seed=SEED,
        )
        raw = resp.choices[0].message.content or ""
    except Exception as e:
        print(f"Error on {row['case_id']}/{row['spell_index']}: {e}")
        raw = ""
    predictions.append(parse_prediction(raw, valid_codes))

predictions_df = test_df[["case_id", "spell_index"]].copy()
predictions_df["predicted_code"] = predictions
predictions_df.head()

In [8]:
predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
    subject  = str(row["subject_label"] or "")
    user_msg = f"Subject description: {subject}"
    try:
        resp = client.chat.completions.create(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_msg},
            ],
            temperature=TEMPERATURE, max_tokens=MAX_TOKENS, seed=SEED,
        )
        raw = resp.choices[0].message.content or ""
        print(raw)
    except Exception as e:
        print(f"Error on {row['case_id']}/{row['spell_index']}: {e}")
        raw = ""
    predictions.append(parse_prediction(raw, valid_codes))

predictions_df = test_df[["case_id", "spell_index"]].copy()
predictions_df["predicted_code"] = predictions
predictions_df.head()

KeyboardInterrupt: 

## 7. Evaluate

In [7]:
import pandas as pd
from corex_eval import evaluate

results = evaluate(
    predictions_df,
    task="annotation",
    variable="uni_subject",
    submit=True,
    experiment_path="/home/tom/projects/corex_eval/experiments/annotation/education/lmstudio_llama33_70b_uni_subject/config.yaml",
)
pd.Series({k: v for k, v in results.items() if k not in ("per_class", "per_country")})

accuracy                       0.0
macro_f1                       0.0
weighted_f1                    0.0
semantic_similarity           None
n_classes                       45
n_evaluated                    952
n_skipped                        0
granularity                   fine
task                    annotation
variable               uni_subject
dtype: object